# Audio ATCOSIM

Sumber: TU Graz / HF Jzuluaga variant. Subset 100 utterances dari satu speaker untuk smoke test fitur; tidak ada fatigue score.

In [ ]:
from pathlib import Path
import os
for key in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMBA_NUM_THREADS'):
    os.environ.setdefault(key, '4')
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
os.chdir(ROOT)
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from atc_fatigue.utils.runtime import load_config
config = load_config()
%matplotlib inline


In [ ]:
manifest = pd.read_parquet('data/manifests/atcosim_subset.parquet')
features = pd.read_parquet('data/interim/atcosim_audio_features.parquet')
display(features[['sample_id', 'subject_id', 'duration_seconds', 'source_sample_rate', 'quality']].head())
print('Utterances:', len(features), 'Speakers:', features.subject_id.nunique())
print('eGeMAPS:', len(features.filter(regex='^egemaps__').columns), 'MFCC:', len(features.filter(regex='^mfcc_').columns))

In [ ]:
import soundfile as sf
signal, rate = sf.read(manifest.original_path.iloc[0])
fig, ax = plt.subplots(figsize=(10, 2.5))
ax.plot(np.arange(len(signal))/rate, signal, lw=0.6)
ax.set(xlabel='Time (s)', ylabel='Amplitude', title='ATCOSIM waveform: real source utterance')
plt.show()

In [ ]:
from atc_fatigue.audio.features import extract_file, feature_names
rerun = extract_file(manifest.original_path.iloc[0], config['audio'])
assert len([k for k in rerun if k.startswith('egemaps__')]) == 88
display(pd.Series(rerun)[feature_names()[:6]])
display(features.filter(regex='^(egemaps__|mfcc_)').isna().mean().describe())

Durasi utterance bukan speech duration hasil VAD. Radio kompresi, noise, serta perbedaan microphone tetap menjadi risiko domain shift.